# Véletlenszerű játszma Stockfish-elemzése + LLM narráció API hívással

**Pipeline:**
1. Véletlenszerű játszma sorsolása a `mychessdotcomgames.parquet` fájlból
2. Stockfish elemzés (centipawn annotációk lépésenként)
3. Annotált PGN exportálása → `output/llm-analysis/elemzett.pgn`
4. LLM API: didaktikus narráció generálása (provider: `config.LLM_PROVIDER`)
5. Narráció mentése → `output/llm-analysis/szoveges/{llm_provider}_{game_id}.txt`
6. TTS hangfájl generálása → `output/llm-analysis/hangos_narracio/{llm_provider}_{game_id}.mp3`

In [1]:
import os
import sys
import random
import shutil

# ROOT_DIR: a könyvtár ahol a config.py van
_cwd = os.getcwd()
ROOT_DIR = _cwd if os.path.exists(os.path.join(_cwd, "config.py"))            else os.path.abspath(os.path.join(_cwd, ".."))
sys.path.insert(0, ROOT_DIR)

import polars as pl
import chess
import chess.pgn
import chess.engine

import config
from src.llm_client import generate_text

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"LLM provider: {config.LLM_PROVIDER}  ({config.LLM_MODEL})")
print(f"API kulcs   : {chr(39)+chr(79)+chr(75)+chr(39) if config.LLM_API_KEY else chr(39)+'HIÁNYZIK – töltsd ki a secrets.py-t!'+chr(39)}")

ROOT_DIR    : D:\Workspace\chess-pgn-analysis
LLM provider: openai  (gpt-4o-mini)
API kulcs   : 'OK'


## 1. Véletlen játszma kiválasztása

In [2]:
PARQUET_PATH = os.path.join(ROOT_DIR, "output", "parquet", "mychessdotcomgames.parquet")

df = pl.read_parquet(PARQUET_PATH)
print(f"Összes játszma a parquet-ban: {len(df):,}")

idx = random.randint(0, len(df) - 1)
row = df.row(idx, named=True)

print(f"\nKiválasztott játszma (sor-index={idx}, game_id={row['game_id']}):")
print(f"  Fehér  : {row['white']} ({row['white_elo']})")
print(f"  Fekete : {row['black']} ({row['black_elo']})")
print(f"  Megnyitó: {row['eco']} · {row['opening']}")
print(f"  Eredmény: {row['result']}  |  Lépések: {row['num_moves']}")

Összes játszma a parquet-ban: 1,377

Kiválasztott játszma (sor-index=489, game_id=490):
  Fehér  : oty_2 (868)
  Fekete : Wujajin (857)
  Megnyitó: D02 · 
  Eredmény: 1-0  |  Lépések: 104


## 2. Stockfish elemzés: heurisztika az LLM-nek!

A kisebb depth gyorsabb futást eredményez, de a magasabb alaposabb (pontosabb) elemzést!

In [3]:
def find_stockfish() -> str:
    if getattr(config, 'STOCKFISH_PATH', None) and os.path.isfile(config.STOCKFISH_PATH):
        return config.STOCKFISH_PATH
    bundled = os.path.join(ROOT_DIR, "stockfish", "stockfish-windows-x86-64-avx2.exe")
    if os.path.isfile(bundled):
        return bundled
    found = shutil.which("stockfish")
    if found:
        return found
    raise FileNotFoundError("Stockfish nem található! Ellenőrizd a stockfish/ mappát.")

SF_PATH = find_stockfish()
print(f"Stockfish: {SF_PATH}")

Stockfish: D:\Workspace\chess-pgn-analysis\stockfish\stockfish-windows-x86-64-avx2.exe


In [4]:
import asyncio
from tqdm.notebook import tqdm

# Windows + Python 3.9 alatt a Jupyter SelectorEventLoop-ot használ,
# ami nem tud subprocesst indítani – a chess.engine ezt igényli.
if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Depth csökkentése gyorsabb futáshoz (config: 18, teszt: 12)
DEPTH       = 12
MOVES_LIMIT = config.STOCKFISH_MOVES_LIMIT

moves_uci_list = row['moves_uci'].strip().split()
to_analyze     = min(len(moves_uci_list), MOVES_LIMIT)
print(f"Elemzés: mélység={DEPTH}, lépések={to_analyze}/{len(moves_uci_list)}\n")

board       = chess.Board()
evaluations = []

with chess.engine.SimpleEngine.popen_uci(SF_PATH) as engine:
    limit = chess.engine.Limit(depth=DEPTH)

    for uci in tqdm(moves_uci_list[:MOVES_LIMIT], desc="Stockfish", unit="lépés"):
        move = chess.Move.from_uci(uci)
        if move not in board.legal_moves:
            print(f"  Illegális lépés: {uci} – leállítás")
            break

        san = board.san(move)
        board.push(move)

        info  = engine.analyse(board, limit)
        score = info["score"].white()

        evaluations.append({
            "uci":  uci,
            "san":  san,
            "cp":   score.score(mate_score=10000),
            "mate": score.mate(),
        })

print(f"\nKész: {len(evaluations)} lépés elemezve.")

Elemzés: mélység=12, lépések=40/104



Stockfish:   0%|          | 0/40 [00:00<?, ?lépés/s]


Kész: 40 lépés elemezve.


## 3. Annotált PGN exportálása

In [5]:
game_id   = row["game_id"]
os.makedirs(config.LLM_ANALYSIS_DIR, exist_ok=True)
PGN_PATH  = config.ELEMZETT_PGN

# Duplikátum-ellenőrzés
pgn_exists     = os.path.exists(PGN_PATH) and os.path.getsize(PGN_PATH) > 0
already_in_pgn = False
if pgn_exists:
    with open(PGN_PATH, encoding="utf-8") as f:
        if f'[ParquetGameId "{game_id}"]' in f.read():
            already_in_pgn = True

if already_in_pgn:
    print(f"⚠️  Játszma #{game_id} már szerepel az elemzett.pgn-ben – kihagyva.")
else:
    game_pgn = chess.pgn.Game()
    game_pgn.headers.update({
        "Event":         row.get("event",    "?"),
        "Site":          row.get("site",     "?"),
        "Date":          row.get("date",     "?"),
        "White":         row.get("white",    "?"),
        "Black":         row.get("black",    "?"),
        "Result":        row.get("result",   "*"),
        "WhiteElo":      str(row.get("white_elo", "?")),
        "BlackElo":      str(row.get("black_elo", "?")),
        "ECO":           row.get("eco",      "?"),
        "Opening":       row.get("opening",  "?"),
        "ParquetGameId": str(game_id),
    })

    node = game_pgn
    for ev in evaluations:
        node = node.add_variation(chess.Move.from_uci(ev["uci"]))
        if ev["mate"] is not None:
            node.comment = f"[%eval #{ev['mate']}]"
        elif ev["cp"] is not None:
            node.comment = f"[%eval {ev['cp'] / 100:.2f}]"

    # StringExporter garantál egységes kimenetet; .strip()+"\n" pontosan egy \n-re zárja a játszmát.
    # Ha már van tartalom, egy "\n" elé kerül → mindig pontosan egy üres sor lesz a játszmák között.
    pgn_str = game_pgn.accept(chess.pgn.StringExporter(headers=True, variations=True, comments=True))
    with open(PGN_PATH, "a", encoding="utf-8") as f:
        if pgn_exists:
            f.write("\n")
        f.write(pgn_str.strip() + "\n")

    with open(PGN_PATH, encoding="utf-8") as f:
        content = f.read()
    game_count = content.count("[Event ")
    print(f"PGN hozzáfűzve: {PGN_PATH}")
    print(f"Összes elemzett játszma: {game_count}")
    print("\nUtolsó hozzáfűzött játszma (első 600 karakter):")
    last_start = content.rfind("[Event ")
    print(content[last_start:last_start + 600])

PGN hozzáfűzve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\elemzett.pgn
Összes elemzett játszma: 2

Utolsó hozzáfűzött játszma (első 600 karakter):
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2025.10.26"]
[Round "?"]
[White "oty_2"]
[Black "Wujajin"]
[Result "1-0"]
[WhiteElo "868"]
[BlackElo "857"]
[ECO "D02"]
[Opening ""]
[ParquetGameId "490"]

1. d4 { [%eval 0.30] } 1... c6 { [%eval 0.46] } 2. Bf4 { [%eval 0.35] } 2... d5
{ [%eval 0.42] } 3. Nf3 { [%eval 0.36] } 3... Bg4 { [%eval 0.62] } 4. e3
{ [%eval 0.48] } 4... e6 { [%eval 0.44] } 5. c3 { [%eval 0.15] } 5... Nd7
{ [%eval 0.40] } 6. Nbd2 { [%eval 0.35] } 6... c5 { [%eval 0.71] } 7. Bd3
{ [%eval 0.42] } 7... c4 { [%eval 0.72] } 8. Bc2 { [%eval 0.63] } 8... Ne7
{ [%eval 0.95] } 9. b


## 4. LLM narráció generálása

In [6]:
NARRATION_PROMPT = (
    "Készíts 2 bekezdésben didaktív szöveges narrációt a csatolt .pgn sakkjátszma alapján, "
    "ügyelj rá, hogy a figurák nevét jól írd ki (mivel hangosan fel lesz olvasva), "
    "amikor egy lépésre hivatkozol, pl. Rxh7-et így írd: bástya üti h7-et, stb.! "
    "A narrációd célja az oktatás és hogy döntő kulcspillanatokat kiemelj!"
)

with open(PGN_PATH, encoding="utf-8") as f:
    pgn_text = f.read()

full_prompt = NARRATION_PROMPT + chr(10) * 2 + pgn_text
narration = generate_text(full_prompt)

print("--- LLM narráció ---")
print(narration)

--- LLM narráció ---
A sakkjátszma első fontos lépései során mindkét fél igyekezett a középső mezők uralmát megszerezni. A fehér hős, Wujajin, az első lépésben e4-et lépett, amit a fekete, jit99, e5-tel válaszolt. Egy klasszikus nyitás kezdődött, ahol a figurák gyorsan gyors harcba bocsátkozta a mezőért. A bástya és a huszár optimális fejlődése érdekében Wujajin a második lépésben Nf3-at lépett, míg Jit99 a saját huszárját vitte Nc6-ra. A negyedik lépés után, amikor a fehér d3-mal biztosította a középpontot, a fekete hirtelen O-o lépéssel bevédte a királyát. Ez a lépés kulcsfontosságú volt, mivel a biztonságos király lehetőséget biztosított a védekezésre, miközben a figurák aktívan részt vettek a harcban.

A játék későbbi szakaszában a fehér áttörést hajtott végre a huszár c6-ra való lépésével, aminek következtében a bástya úgy döntött, hogy megüti az E2-t, biztosítva magának egy döntő előnyt. Ekkor a fekete bástya is gyorsan lépett, hogy mentse a pozícióját, de a fehér hős a huszár és

## 5. Narráció mentése

In [7]:
from tqdm.notebook import tqdm

llm_name = config.LLM_PROVIDER
txt_name = f"{llm_name}_{game_id}.txt"
os.makedirs(config.LLM_ANALYSIS_SZOVEGES_DIR, exist_ok=True)
LLM_PATH = os.path.join(config.LLM_ANALYSIS_SZOVEGES_DIR, txt_name)

with tqdm(total=2, desc="Narráció mentése", unit="lépés") as pbar:
    pbar.set_postfix_str("duplikátum-ellenőrzés")
    already_exists = os.path.exists(LLM_PATH)
    pbar.update(1)

    if already_exists:
        pbar.set_postfix_str("kihagyva")
        pbar.update(1)
        print(f"⚠️  Narráció már létezik: {LLM_PATH} – kihagyva.")
    else:
        pbar.set_postfix_str("fájl írás")
        with open(LLM_PATH, "w", encoding="utf-8") as f:
            f.write(narration)
        pbar.update(1)
        print(f"Narráció elmentve: {LLM_PATH}")

Narráció mentése:   0%|          | 0/2 [00:00<?, ?lépés/s]

Narráció elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\szoveges\openai_490.txt


## 6. TTS hangfájl generálása (kicsit lassabban fut!)

In [8]:
from src.tts_client import generate_audio

mp3_name = txt_name.replace(".txt", ".mp3")
os.makedirs(config.LLM_ANALYSIS_HANGOS_DIR, exist_ok=True)
MP3_PATH = os.path.join(config.LLM_ANALYSIS_HANGOS_DIR, mp3_name)

if os.path.exists(MP3_PATH):
    print(f"⚠️  Hangfájl már létezik: {MP3_PATH} – kihagyva.")
else:
    with open(LLM_PATH, encoding="utf-8") as f:
        narration_text = f.read()
    generate_audio(narration_text, MP3_PATH, show_progress=True)
    print(f"Hangfájl elmentve: {MP3_PATH}  ({os.path.getsize(MP3_PATH):,} bájt)")

TTS letöltés (OpenAI): 0.00B [00:00, ?B/s]

Hangfájl elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\hangos_narracio\openai_490.mp3  (1,779,840 bájt)
